# Xây dựng Knowledge Base từ nguồn PDF

Audit nguồn → corpus bất biến → BGE-M3 dense/sparse → PostgreSQL/pgvector → graph và kiểm chứng. Gold Dev/Test không được ingest. Nguồn PDF xác lập context; hiệu lực pháp lý vẫn là metadata nguồn local. Đánh giá luồng chung HS/Legal nằm trong notebook 02.


In [ ]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/chunking/corpus.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from config.retrieval import retrieval_config  # noqa: E402 - project root must be resolved first

cfg = retrieval_config()
print("Workspace:", ROOT)

In [ ]:
import time

import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.ingestion.audit import audit
from rag.chunking.corpus import build_corpus
from rag.vectordb.postgres import connect, counts, ingest, migrate
from rag.embedding.bge_m3 import Embedder, load_embeddings

## Đối chiếu dữ liệu nguồn sau chuẩn hoá

Kiểm tra schema v2 và văn bản tìm kiếm tiếng Việt theo từng bản ghi. Các sửa OCR lịch sử đã nằm trong `data/normalized`; bước này chỉ đọc dữ liệu và không ghi lại nguồn.


In [ ]:
from rag.ingestion.sources import source_files, tariff_bundle
from rag.ingestion.sources import source_manifest

registered = source_manifest(ROOT)
data = source_files()
for source in registered["tariffs"]:
    records, nodes, edges = tariff_bundle(source, data)
    assert len({r["id"] for r in records}) == len(records)
    assert all(r["locator"] and r["source_text"] for r in records)
    print(source["records"], len(records), len(nodes), len(edges), "source adapter: PASS")


## 1. Audit và truy vết nguồn
TT31 ingest được nối với master; GIR combined và TT85 source là allowlist. data/verified/hs gồm Dev và Test được dùng kiểm coverage mã, không đưa vào corpus.

In [ ]:
from rag.ingestion.sources import NORMALIZED, SOURCES

audit_result = audit()
display(pd.DataFrame(audit_result["quality"]).T[["count", "duplicate_wording"]])
display(
    pd.DataFrame(
        [
            {
                "scope_chapters": audit_result["scope_count"],
                "graph_edges": audit_result["graph_edges"],
                "broken_endpoints": len(audit_result["broken_endpoints"]),
                "gold_codes_missing": len(audit_result["gold_missing"]),
                "TT85_SHA256_match": audit_result["tt85_source_hash_matches"],
                "legal_verification": audit_result["legal_verification"],
            }
        ]
    )
)



display(pd.DataFrame([
    {"records": source["records"], "edges": source.get("edges"), "nodes": source.get("nodes")}
    for source in registered["tariffs"]
]))
print("Only manifest-registered sources enter ingestion.")


## 2. Corpus và metadata thống nhất
GIR giữ source atomic và context view; TT85 giữ điều khoản, loại 5 forms khỏi RAG. Structural/context/footnote chunks giữ nguyên vai trò và chunk_type=null khi không thuộc 7 loại statutory/tariff đã chốt, tránh đổi tên footnote thành điều luật. Scope index ngoài 3 Sections chỉ hỗ trợ nhận biết phạm vi.


In [ ]:
bundle = build_corpus()
records = pd.DataFrame(bundle["records"])
display(
    records.groupby(["namespace", "dataset", "section", "role"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
display(
    records.groupby(["chunk_type", "level"], dropna=False).size().rename("records").reset_index()
)
print("Snapshot:", bundle["snapshot"], "Corpus:", len(records))
assert len(records) == bundle["manifest"]["count"]
assert records["id"].is_unique
assert not records["id"].str.contains("TT85-FORM-").any()
assert set(records["namespace"]) == {"hs_classification", "legal_classification"}

node_ids = {n["id"] for n in bundle["nodes"]}
assert all(e["source"] in node_ids and e["target"] in node_ids for e in bundle["edges"])
assert all(r["locator"] for r in bundle["records"])
print("Source hashes:", bundle["manifest"]["source_hashes"])
print("Graph:", len(bundle["nodes"]), "nodes /", len(bundle["edges"]), "edges")


In [ ]:
from rag.ingestion.sources import NORMALIZED
from rag.utils.files import read

examples = [
    r
    for r in bundle["records"]
    if r["id"] in ["gir:GIR_3_NOTE_038", "gir:GIR_5_NOTE_016", "tt85:TT85-A9-3.a.1"]
]
display(
    pd.DataFrame(
        [
            {
                "id": r["id"],
                "source_text": r["source_text"],
                "embedding_view": r["embedding_text"],
                "locator": r["locator"],
            }
            for r in examples
        ]
    )
)

forms = read(NORMALIZED / "TT85/tt85_forms_source.json")
display(
    pd.DataFrame(
        [
            {
                "form": f["form_no"],
                "code": f["code"],
                "title": f["title"],
                "fields": len(f["fields"]),
            }
            for f in forms
        ]
    )
)
print("5 forms are a template registry, not RAG chunks; Word generation is not implemented.")
assert len(forms) == 5

## 3. BGE-M3 dense + sparse
Pin model/tokenizer commit. Corpus input không được truncate im lặng. Checkpoint theo batch/input hash giúp tái chạy nhanh. CPU FP32 và sparse là cấu hình đã chọn cho experiment; dense là 1.024 chiều, sparse là token weights của model.

In [ ]:
embedder = Embedder(cfg)
started = time.perf_counter()
with capture_output() as embedding_progress:
    embedding_manifest = embedder.corpus(bundle)
manifest, vectors, sparse = load_embeddings(bundle)
display(
    pd.DataFrame(
        [
            {
                "records": len(vectors),
                "dimension": vectors.shape[1],
                "max_input_tokens": manifest["max_tokens"],
                "sparse_nonzero": sum(len(w) for w in sparse),
                "elapsed_seconds": round(time.perf_counter() - started, 2),
                "model_revision": manifest["profile"]["revision"],
                "profile": manifest["profile_id"],
            }
        ]
    )
)
np.testing.assert_allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5)
assert len(sparse) == len(vectors)
assert np.isfinite(vectors).all()

## 4. PostgreSQL: nạp snapshot đã kiểm chứng

Mặc định giữ snapshot cũ và kích hoạt snapshot mới trong transaction. `FRESH_REBUILD=True` vẫn là lựa chọn xóa schema RAG khi người vận hành chủ động chọn; không cần để thêm chương. Embeddings không đổi được tái sử dụng theo hash.


In [ ]:
conn = connect()
migrate(conn)
display(pd.DataFrame([conn.execute("SELECT current_database() AS database, version() AS server").fetchone()]))
APPLY = True
FRESH_REBUILD = False
RAG_TABLES = (
    "retrieval_results", "evaluation_runs", "active", "sparse_postings",
    "embeddings", "graph_edges", "graph_nodes", "chunks", "snapshots", "profiles",
)
actual_tables = {row["table_name"] for row in conn.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema='rag' AND table_type='BASE TABLE'"
).fetchall()}
assert actual_tables == set(RAG_TABLES), (actual_tables, RAG_TABLES)
before_rebuild = {table: conn.execute(f"SELECT count(*) AS n FROM rag.{table}").fetchone()["n"]
                  for table in RAG_TABLES}
print("RAG database before fresh rebuild:", before_rebuild)
print("Apply:", APPLY, "fresh rebuild:", FRESH_REBUILD)


In [ ]:
if APPLY and FRESH_REBUILD:
    # All statements run on the isolated RAG schema; no CASCADE into other schemas.
    with conn.transaction():
        conn.execute("SELECT pg_advisory_xact_lock(20260926)")
        conn.execute("TRUNCATE TABLE " + ", ".join(f"rag.{table}" for table in RAG_TABLES))
        assert all(conn.execute(f"SELECT count(*) AS n FROM rag.{table}").fetchone()["n"] == 0
                   for table in RAG_TABLES)
        print("Old RAG tables cleared inside transaction.")
        ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=True)
        assert conn.execute("SELECT count(*) AS n FROM rag.snapshots").fetchone()["n"] == 1
        assert conn.execute("SELECT count(*) AS n FROM rag.chunks").fetchone()["n"] == len(bundle["records"])
        assert conn.execute("SELECT snapshot FROM rag.active").fetchone()["snapshot"] == bundle["snapshot"]
else:
    ingestion_result = ingest(conn, bundle, manifest, vectors, sparse, apply=APPLY)
display(pd.DataFrame([ingestion_result.get("counts", ingestion_result.get("expected"))]))
if APPLY:
    after = counts(conn, bundle["snapshot"], manifest["profile_id"])
    assert after == ingestion_result["counts"]
    print("Fresh snapshot and counts: PASS")
    display(pd.DataFrame([conn.execute("SELECT * FROM rag.active").fetchone()]))


## 5. Kiểm chứng cuối
Counts, FK integrity, vector norms và sparse coverage đều được kiểm trực tiếp. Phần QA phía dưới và notebook 02 dùng snapshot/profile active này; không cần giữ kernel hiện tại.

In [ ]:
if APPLY:
    check = conn.execute(
        """SELECT count(*) AS sparse_postings FROM rag.sparse_postings
        WHERE snapshot=%s AND profile=%s""",
        (bundle["snapshot"], manifest["profile_id"]),
    ).fetchone()
    assert check["sparse_postings"] == sum(len(w) for w in sparse)
    print(
        "Knowledge base ready:", {"snapshot": bundle["snapshot"], "records": len(vectors), **check}
    )
conn.close()

## Definition graph audit for retrieval
The active knowledge base already contains Notes and definition edges. Derive cited search views from that active snapshot; do not re-ingest gold or change atomic source wording. Search views are built in memory for BM25 and optional evidence-aware reranking. Existing BGE dense/sparse vectors remain valid. Notebook 02 benchmarks and freezes this retrieval policy.

In [ ]:
import pandas as pd
from IPython.display import display

from rag.vectordb.postgres import connect
from rag.retrieval.retriever import Retriever

with connect() as audit_connection:
    definition_audit = Retriever(audit_connection)
    definition_views = definition_audit.definition_views()
    active_before = audit_connection.execute("SELECT * FROM rag.active").fetchone()
    display(pd.DataFrame([{"section": view["record"]["section"],
        "heading": view["record"]["heading"], "ahtn8": view["record"]["code"],
        "source_ids": [x["id"] for x in view["citations"]],
        "graph_paths": view["paths"]} for view in definition_views.values()]))
    assert all(view["record"]["source_text"] == definition_audit.exact(view["record"]["code"])[0]["source_text"]
        for view in definition_views.values())
    assert audit_connection.execute("SELECT * FROM rag.active").fetchone() == active_before
    print("Source-backed definition views:", len(definition_views), "Active snapshot:", active_before)


In [ ]:
conn.close()


## Đánh giá retrieval sau khi build

Dùng `02_test_retrieval.ipynb` để đọc báo cáo v5 và thử `Retriever.routed_search`. HS và Legal dùng cùng cấu hình `config/unified_retrieval.json`; không chọn riêng policy QA trong notebook build.

Cấu hình đã khóa gắn với snapshot và embedding profile. Nếu nguồn/model đổi, cần một lượt thực nghiệm mới trên Dev trước khi kích hoạt cấu hình tương ứng; không tự sửa hash để bỏ qua kiểm tra. Các runner v4 chỉ phục vụ tái lập đối chứng.
